In [ ]:
%matplotlib notebook
%matplotlib widget

import sys
sys.path.append('../../')
import config

from netCDF4 import Dataset, num2date
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## About the data
The air temperatures are from May - November 2021. 
For the month of May there is only data for the 30th and 31st May.
For the rest of the months it is for the entire month.

### Time

The time is in the units: "hours since 2000-01-01 00:00:00 UTC" 

Values: [187680, 187681, 187682, 187683, 187684, ...] meaning it is a count of hours since midnight on January 1, 2000. 

Therefore 187680 = Jan 1, 2000

### Temperature data

The temperature has the shape (48, 280, 386) where dimensions: (time, latitude, longitude).

### Aim
For the dataset find periods of
- 3 days above 30 degrees
- 5 days above 25 degrees
- or langdurig (25 degrees) 

In [ ]:
# This is the file for data: 2021-05-30 00:00:00 to 2021-05-31 23:00 
temp_dir = config.KNMI_TEMP_DIR

In [ ]:
# Open the NetCDF file
temp_file = Dataset(temp_dir, mode='r')

# See what variables are inside
print(temp_file.variables.keys())

print(temp_file.variables['longitude'])
print(temp_file.variables['latitude'])
print(temp_file.variables['air_temperature'])


In [ ]:
temp_file

In [ ]:
# Get the variable
air_temp_var = temp_file.variables['air_temperature']

# Read raw data
raw_data = air_temp_var[:]

# Mask out fill values
fill_value = air_temp_var._FillValue
masked_data = np.ma.masked_equal(raw_data, fill_value)

# Apply scale factor
# The scale factor is 0.01f but we want to only keep the number
scale_factor = 0.01
air_temperature = masked_data * scale_factor

# Now, air_temperature has correct values
print(air_temperature.shape)
print(air_temperature[0, :5, :5])  # Print a small slice for the first time step

In [ ]:
# Open NetCDF file
nc_file = Dataset(temp_dir, mode='r')

# Read time and decode
time_var = nc_file.variables['time']
times = num2date(time_var[:], units=time_var.units, calendar=getattr(time_var, 'calendar', 'standard'))

# Manually cast cftime objects to datetime
times = np.array([pd.Timestamp(t.strftime('%Y-%m-%d %H:%M:%S')) for t in times])


# Read air_temperature and apply scaling
air_temp_var = nc_file.variables['air_temperature']
raw_data = air_temp_var[:]
fill_value = air_temp_var._FillValue
masked_data = np.ma.masked_equal(raw_data, fill_value)
scale_factor = float(str(air_temp_var.scale_factor).replace('f', ''))
air_temperature = masked_data * scale_factor  # Now in degrees Celsius

nc_file.close()

# Now, air_temperature shape = (time, lat, lon)

# Step 1: Take the max temperature across all grid points at each time
max_temp_per_time = air_temperature.max(axis=(1, 2))

# Step 2: Create a pandas DataFrame
df = pd.DataFrame({
    'time': times,
    'max_temp': max_temp_per_time.filled(np.nan)  # convert masked to NaN
})
df.set_index('time', inplace=True)

# Step 3: Resample to daily max
daily_max = df['max_temp'].resample('D').max()
print(daily_max)
# Step 4: Find where daily max > 30 degrees
above_30 = daily_max > 30

# Step 5: Find sequences of 3 or more consecutive days
periods = []
current_period = []

for date, is_above in above_30.items():
    if is_above:
        current_period.append(date)
        if len(current_period) >= 3:
            # Only append if the sequence is at least 3 days long
            if len(periods) == 0 or current_period[0] != periods[-1][-1]:
                periods.append(current_period.copy())
    else:
        current_period = []

# Step 6: Display results
for period in periods:
    print(f"Period from {period[0].date()} to {period[-1].date()}")



## Test plot

In [ ]:
# Load the dataset
ds = Dataset(temp_dir, mode='r')

# Extract variables
lat = ds.variables['latitude'][:]
lon = ds.variables['longitude'][:]
temp = ds.variables['air_temperature']

# Attributes
scale_factor = getattr(temp, 'scale_factor', 1.0)
fill_value = getattr(temp, '_FillValue', -9999.0)

# Data at first time index
temp_raw = temp[0, :, :]
scale_factor = float(str(scale_factor).replace('f', ''))  # Convert to float if necessary


temp_data = np.array(temp_raw, dtype='float32') * scale_factor
temp_data = np.where(temp_data == fill_value * scale_factor, np.nan, temp_data)

# Create meshgrid
lon2d, lat2d = np.meshgrid(lon, lat)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
c = ax.pcolormesh(lon2d, lat2d, temp_data, shading='auto', cmap='coolwarm')
fig.colorbar(c, ax=ax, label='Air Temperature (°C)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Air Temperature (Time Index 0)')
plt.show()

In [ ]:
# Load dataset
ds = Dataset(temp_dir, mode='r')

# Get variables
lat = ds.variables['latitude'][:]
lon = ds.variables['longitude'][:]
temp = ds.variables['air_temperature']

# Get attributes
scale_factor = getattr(temp, 'scale_factor', 1.0)
fill_value = getattr(temp, '_FillValue', -9999.0)

# Extract and process data for time index 0
scale_factor = float(str(scale_factor).replace('f', ''))  # Convert to float if necessary
temp_raw = np.array(temp[0, :, :], dtype='float32') * scale_factor
temp_data = np.where(temp_raw == fill_value * scale_factor, np.nan, temp_raw)

# Flatten for scatter plot
lat2d, lon2d = np.meshgrid(lat, lon, indexing='ij')  # match shape
lat_flat = lat2d.flatten()
lon_flat = lon2d.flatten()
temp_flat = temp_data.flatten()

# Filter out NaNs
mask = ~np.isnan(temp_flat)
lat_plot = lat_flat[mask]
lon_plot = lon_flat[mask]
temp_plot = temp_flat[mask]

# Plot as scatter
fig_1, ax_1 = plt.subplots(figsize=(8, 6))
sc = ax_1.scatter(lon_plot, lat_plot, c=temp_plot, cmap='coolwarm', s=10, alpha=0.8)
plt.colorbar(sc, ax=ax_1, label='Air Temperature (°C)')
ax_1.set_xlabel('Longitude')
ax_1.set_ylabel('Latitude')
ax_1.set_title('Air Temperature as Points (Time Index 0)')
plt.show()
